# Build `karl4th/limmim-v2` from raw LibriSpeech through Mimi

This notebook replaces the old filtered `karl4th/limmim` cache. It processes every utterance in:

- `clean/train.100 + clean/train.360` — exactly 132,553 source rows
- `clean/validation` — exactly 2,703 source rows
- `clean/test` — exactly 2,620 source rows

No duration filter and no CTC-feasibility filter are applied. Every output row contains the exact LibriSpeech transcript, Mimi q0 semantic codes, and the transcript's UTF-8 byte target. Output is written as restartable Parquet shards directly to Google Drive. An interrupted run resumes from the last atomic shard.

## 0. Mount Google Drive

In [ ]:
import logging, os, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", force=True)
print("DRIVE: mounted", flush=True)

## 1. Checkout the exact `stage2` branch

In [ ]:
REPO_URL = "https://github.com/karl4th/aether-v3.git"
REPO_DIR = "/content/aether-v3"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", "stage2"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", "stage2", "--single-branch", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "stage2"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/stage2"], check=True)
os.chdir(REPO_DIR)
print("CODE:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(), flush=True)

## 2. Install the extraction dependencies

In [ ]:
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR,
    "transformers>=5.17", "datasets>=2.19,<4.0", "soundfile", "librosa",
    "pyyaml", "numpy", "tqdm", "huggingface_hub", "hf_transfer", "pyarrow",
], check=True)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
sys.path.insert(0, str(Path(REPO_DIR)/"src"))
for module_name in list(sys.modules):
    if module_name == "aether_v3" or module_name.startswith("aether_v3."):
        del sys.modules[module_name]
print("DEPENDENCIES: ready", flush=True)

## 3. Read the Hugging Face token

In [ ]:
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
assert os.environ["HF_TOKEN"], "Add HF_TOKEN in Colab Secrets and enable notebook access"
print("HF TOKEN: available (value hidden)", flush=True)

## 4. Fix the source and destination contract

In [ ]:
import torch
from aether_v3.config import load_config

cfg = load_config("configs/ctc_base.yaml")
cfg.mimi.num_quantizers = 1
cfg.data.extraction_batch_size = 32
cfg.data.extraction_num_workers = max(2, min(os.cpu_count() or 4, 16))
OUTPUT_ROOT = Path("/content/drive/MyDrive/aether-v3/datasets/limmim-v2")
HF_REPO_ID = "karl4th/limmim-v2"
EXPECTED = {"train": 132553, "validation": 2703, "test": 2620}
assert torch.cuda.is_available(), "Select a GPU runtime"
print("GPU:", torch.cuda.get_device_name(0), flush=True)
print("OUTPUT:", OUTPUT_ROOT, flush=True)
print("MIMI:", cfg.mimi.pretrained_id, "q0 only", flush=True)

## 5. Download all raw LibriSpeech splits

In [ ]:
from aether_v3.data.mimi_cache import download_raw_splits

print("DOWNLOAD: train.100 + train.360 + validation + test", flush=True)
download_raw_splits(cfg)
print("DOWNLOAD: complete", flush=True)

## 6. Load the raw splits and verify their exact counts

In [ ]:
from datasets import concatenate_datasets
from aether_v3.data.librispeech import load_split

def load_role(specs):
    parts = []
    for spec in specs:
        part = load_split(spec, cfg.data.dataset_id, cfg.data.fallback_dataset_id)
        part = part.add_column("source_split", [spec] * len(part))
        parts.append(part)
    return parts[0] if len(parts) == 1 else concatenate_datasets(parts)

raw_train = load_role(cfg.data.train_splits)
raw_validation = load_role(cfg.data.validation_splits)
raw_test = load_role(cfg.data.test_splits)
raw_by_role = {"train": raw_train, "validation": raw_validation, "test": raw_test}
specs_by_role = {
    "train": cfg.data.train_splits,
    "validation": cfg.data.validation_splits,
    "test": cfg.data.test_splits,
}
actual = {role: len(dataset) for role, dataset in raw_by_role.items()}
assert actual == EXPECTED, f"Unexpected LibriSpeech counts: {actual} != {EXPECTED}"
print("RAW COUNTS:", actual, flush=True)
print("SAMPLE:", raw_train[0]["source_split"], raw_train[0]["text"], flush=True)


## 7. Extract the complete training split through Mimi

The progress display is one live line with processed utterances, speed and ETA. A Parquet shard is committed every 2,048 rows; rerunning this cell resumes automatically.

In [ ]:
from aether_v3.data.limmim_v2 import build_limmim_v2_split

train_manifest = build_limmim_v2_split(
    raw_train,
    cfg.mimi,
    OUTPUT_ROOT/"train",
    "train",
    cfg.data.train_splits,
    device="cuda",
    batch_size=cfg.data.extraction_batch_size,
    num_workers=cfg.data.extraction_num_workers,
    shard_size=2048,
)
assert train_manifest["examples"] == EXPECTED["train"]
print("TRAIN COMPLETE:", train_manifest, flush=True)

## 8. Extract the complete validation split through Mimi

In [ ]:
validation_manifest = build_limmim_v2_split(
    raw_validation,
    cfg.mimi,
    OUTPUT_ROOT/"validation",
    "validation",
    cfg.data.validation_splits,
    device="cuda",
    batch_size=cfg.data.extraction_batch_size,
    num_workers=cfg.data.extraction_num_workers,
    shard_size=2048,
)
assert validation_manifest["examples"] == EXPECTED["validation"]
print("VALIDATION COMPLETE:", validation_manifest, flush=True)

## 9. Extract the complete test split through Mimi

In [ ]:
test_manifest = build_limmim_v2_split(
    raw_test,
    cfg.mimi,
    OUTPUT_ROOT/"test",
    "test",
    cfg.data.test_splits,
    device="cuda",
    batch_size=cfg.data.extraction_batch_size,
    num_workers=cfg.data.extraction_num_workers,
    shard_size=2048,
)
assert test_manifest["examples"] == EXPECTED["test"]
print("TEST COMPLETE:", test_manifest, flush=True)

## 10. Audit every shard and verify transcript/byte integrity

In [ ]:
import json, random
import pyarrow.parquet as pq

summary = {}
for role, expected in EXPECTED.items():
    paths = sorted((OUTPUT_ROOT/role).glob("part-*.parquet"))
    count = sum(pq.ParquetFile(path).metadata.num_rows for path in paths)
    assert count == expected, f"{role}: {count} != {expected}"
    summary[role] = {"examples": count, "shards": len(paths)}
    for path in random.Random(1337).sample(paths, min(3, len(paths))):
        table = pq.read_table(path, columns=["transcript", "byte_target", "semantic_codes", "semantic_length"])
        for row in table.slice(0, min(5, len(table))).to_pylist():
            assert bytes(row["byte_target"]).decode("utf-8") == row["transcript"]
            assert len(row["semantic_codes"]) == row["semantic_length"] > 0
manifest = {
    "format": "limmim-v2",
    "source": cfg.data.dataset_id,
    "mimi_model_id": cfg.mimi.pretrained_id,
    "semantic_codebook": 0,
    "filters": [],
    "splits": summary,
}
(OUTPUT_ROOT/"manifest.json").write_text(json.dumps(manifest, indent=2))
print("INTEGRITY AUDIT: PASS", json.dumps(manifest, indent=2), flush=True)

## 11. Load the completed Parquet dataset locally

In [ ]:
from datasets import load_dataset

data_files = {
    role: [str(path) for path in sorted((OUTPUT_ROOT/role).glob("part-*.parquet"))]
    for role in EXPECTED
}
limmim_v2 = load_dataset("parquet", data_files=data_files)
assert {role: len(limmim_v2[role]) for role in EXPECTED} == EXPECTED
print(limmim_v2)
print(limmim_v2["train"].features)

## 12. Publish `karl4th/limmim-v2` to Hugging Face

Run only after the integrity audit passes. This creates a separate dataset repository; it does not overwrite the old filtered `karl4th/limmim`.

In [ ]:
from huggingface_hub import login

login(token=os.environ["HF_TOKEN"])
limmim_v2.push_to_hub(HF_REPO_ID, private=False, token=os.environ["HF_TOKEN"])
print(f"PUBLISHED: https://huggingface.co/datasets/{HF_REPO_ID}", flush=True)

## 13. Final handoff

In [ ]:
print("LIMMIM-V2 READY", flush=True)
print("Drive:", OUTPUT_ROOT, flush=True)
print("Hub:", f"https://huggingface.co/datasets/{HF_REPO_ID}", flush=True)
print("Counts:", EXPECTED, flush=True)